# Лабораторна робота № 5
## Географічні дані OpenStreetMap: від тегів до маршруту

**Інтелектуальний аналіз та візуалізація даних** · 4 години · 4 бали

---

### Постановка

Ви — аналітик у туристичному застосунку. Завдання: для заданого міста
**відібрати десять найвартісніших історико-культурних об'єктів** і
**спланувати піший маршрут**, який починається на залізничному вокзалі,
обходить усі десять і повертається на вокзал.

Слово «найвартісніші» в даних не записане. Критерій ви побудуєте самі —
і захистите його.

Потім те саме для **тридцяти** об'єктів. Результат цієї, більшої задачі
стане базою порівняння в **лабораторній роботі № 8**, де ту саму задачу
розв'язуватимуть генетичним та мурашиним алгоритмами.

### Що здаєте

1. Цей ноутбук із виконаними комірками (усі виводи збережені).
2. Файл `lr5_результати.json` — результати для ЛР8.
3. Карту `lr5_маршрут.html`.
4. Звіт у кінці ноутбука: відповіді на сім питань.

### Що оцінюється

| критерій | 0 | 2 | 3 | 4 |
|---|---|---|---|---|
| Відбір об'єктів | теги взято навмання | фільтр працює | фільтр обґрунтовано | показано, що відкинуто і чому |
| Критерій рейтингу | немає | є формула | ваги пояснено | порівняно з альтернативним критерієм |
| Маршрут | не будується | будується | перевірено коректність | пояснено різницю методів числом |
| Карта | немає | точки на карті | маршрут по ребрах графа | видно різницю з прямими відрізками |
| Звіт | немає | формально | з числами | з висновком, якого не було в завданні |

---
## 0. Ваш варіант

Впишіть прізвище та номер у списку групи. Місто підтягнеться само.

In [ ]:
STUDENT_NAME = 'Ореньчук Юрій'   # <-- ВПИШІТЬ СВОЄ
LIST_NUMBER  = 15     # <-- ВПИШІТЬ СВІЙ НОМЕР У СПИСКУ ГРУПИ

CITIES = [
    'Львів', 'Київ', 'Харків', 'Одеса', 'Дніпро',
    'Чернівці', 'Івано-Франківськ', 'Тернопіль', 'Ужгород', 'Луцьк',
    'Рівне', 'Хмельницький', 'Вінниця', 'Житомир', 'Чернігів',
    'Черкаси', 'Полтава', 'Суми', 'Кропивницький', 'Запоріжжя',
    'Миколаїв', 'Кривий Ріг', 'Кам\u2019янець-Подільський', 'Мукачево',
    'Коломия', 'Дрогобич', 'Біла Церква', 'Ніжин', 'Остріг', 'Умань',
]

assert STUDENT_NAME and LIST_NUMBER > 0, 'впишіть прізвище та номер у списку'
CITY = CITIES[(LIST_NUMBER - 1) % len(CITIES)] + ', Україна'
print('варіант %d: %s' % (LIST_NUMBER, CITY))

варіант 15: Чернігів, Україна


In [2]:
import importlib.util, subprocess, sys
REQUIRED = ['osmnx', 'geopandas', 'shapely', 'pyproj', 'pyarrow',
            'folium', 'mapclassify', 'h3', 'networkx']
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print('встановлюю:', ', '.join(missing), '...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('усі пакети на місці')

усі пакети на місці


In [3]:
import warnings; warnings.filterwarnings('ignore')
import json, time, os, collections
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import folium
from xyzservices import providers as xyz

BASEMAP = xyz.Esri.WorldGrayCanvas   # CARTO з 2026 р. вимагає ключ API
pd.set_option('display.width', 200)
print('geopandas', gpd.__version__)

METRIC_CRS = 32636   # UTM 36N: Чернігів лежить на 31,3° сх.д. (зона 36N: 30–36°). Усі метри — лише в цій СК.

geopandas 1.2.0


---
## 1. Два способи дістати дані

Робоча програма вимагає порівняти два способи доступу **за обсягом переданих
даних і за часом**:

* **живий запит** — `osmnx` питає Overpass API просто зараз;
* **знімок** — ті самі дані, збережені у файли GeoParquet, читаються з диска.

Нижче — живий запит. Запустіть його **один раз**: він збереже теку зі знімком,
і всі наступні запуски читатимуть з неї.

> Overpass — безкоштовний сервіс і обмежує частоту. Якщо запит не пройшов —
> зачекайте хвилину й повторіть. Якщо не проходить взагалі, візьміть
> резервний знімок у викладача і вкажіть це у звіті.

In [4]:
import osmnx as ox

OUT = 'chernihiv_walk2'   # тека зі знімком (зібрана з .osm.pbf скриптом build_city_from_pbf.py)
os.makedirs(OUT, exist_ok=True)
ox.settings.use_cache = True
ox.settings.cache_folder = os.path.join(OUT, 'cache')
ox.settings.requests_timeout = 300
ox.settings.overpass_rate_limit = True

TOURISM_OK = ['museum', 'attraction', 'gallery', 'artwork', 'viewpoint']

def cache_bytes():
    import pathlib
    p = pathlib.Path(ox.settings.cache_folder)
    return sum(f.stat().st_size for f in p.glob('*.json')) if p.exists() else 0

print('тека знімка:', OUT)

тека знімка: chernihiv_walk2


### Дві налаштовані дрібниці, без яких запит не пройде

Це не формальності — кожна з них ламає збір по-своєму, і жодна не схожа
на свою справжню причину.

**Представитись.** Overpass відповідає **HTTP 406** клієнтові, який прийшов
із типовим заголовком `python-requests`. `osmnx` надсилає свій `User-Agent`
сам, але якщо ви звертаєтесь до сервісу напряму — надсилайте теж.
Рядок має бути **лише латиницею**: HTTP-заголовки передаються в latin-1,
і кирилиця кладе запит помилкою `UnicodeEncodeError` ще до відправлення.

**Обрати машину.** За іменем `overpass-api.de` стоїть кілька серверів, і
частина з них відмовляє у з’єднанні. `osmnx` закріплює одну адресу на
всю сесію — навмисно, щоб не порушувати облік слотів. Якщо закріпиться та,
що відмовляє, впаде весь збір. Тому адресу обираємо самі, вже перевірену.

In [5]:
import socket

USER_AGENT = ('LvivPolytechnic-SK26-coursework/1.0 '
              '(https://lpnu.ua; student lab assignment)')
assert USER_AGENT.isascii(), 'HTTP-заголовки — latin-1, кирилиця тут не працює'
ox.settings.http_user_agent = USER_AGENT


def pin_working_ip(hostname, port=443, timeout=8):
    """Закріпити для цього імені адресу, яка справді відповідає."""
    try:
        infos = socket.getaddrinfo(hostname, port, socket.AF_INET, socket.SOCK_STREAM)
    except Exception as e:
        print('не вдалось резолвити %s: %s' % (hostname, type(e).__name__))
        return None
    ips = sorted({i[4][0] for i in infos})
    good = []
    for ip in ips:
        s = socket.socket(); s.settimeout(timeout)
        try:
            s.connect((ip, port)); good.append(ip)
        except Exception:
            pass
        finally:
            s.close()
    print('%s: машин %d, відповідають %d %s' % (hostname, len(ips), len(good), good))
    if not good:
        return None
    original = socket.gethostbyname
    socket.gethostbyname = lambda h: good[0] if h == hostname else original(h)
    print('закріплено', good[0])
    return good[0]


pin_working_ip('overpass-api.de')

не вдалось резолвити overpass-api.de: gaierror


> Якщо жодна машина не відповіла — виконайте комірку ще раз за хвилину,
> або візьміть у викладача резервний знімок і зазначте це у звіті.
> Інші дзеркала перелічені на сторінці
> [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API);
> адресу можна підставити у `ox.settings.overpass_url`.

### Завдання 1.1

Допишіть функцію `fetch`: вона має зберегти шар у `.parquet`, а якщо файл
уже є — прочитати його, **не звертаючись до мережі**. Поверніть також,
скільки МБ пішло мережею і скільки секунд це зайняло.

Підказка щодо `clean`: у OSM одна колонка може містити і число, і список
(ребро, склеєне з кількох шляхів). Parquet вимагає один тип на колонку,
тому неоднорідну колонку треба звести до тексту **цілком**.

In [6]:
def clean(gdf):
    g = gdf.reset_index()
    for c in g.columns:
        if c == 'geometry':
            continue
        boxed = g[c].map(lambda v: isinstance(v, (list, dict, set, tuple))).any()
        if boxed or g[c].dtype == object:
            g[c] = g[c].map(
                lambda v: None if v is None or (isinstance(v, float) and pd.isna(v))
                else (json.dumps(v, ensure_ascii=False)
                      if isinstance(v, (list, dict, set, tuple)) else str(v))
            ).astype('string')
    return g


COST = {}

def fetch(name, getter):
    """Повертає шар. Якщо файл уже збережено — читає з диска."""
    path = os.path.join(OUT, name + '.parquet')
    if os.path.exists(path) and os.path.getsize(path) > 0:
        t0 = time.time()
        g = gpd.read_parquet(path)
        COST[name] = {'спосіб': 'знімок', 'записів': len(g),
                      'передано_МБ': 0.0,
                      'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                      'секунд': round(time.time() - t0, 3)}
        return g

    t0, b0 = time.time(), cache_bytes()          # 1) запам'ятали стан ДО запиту
    g = clean(getter())                          # 2) живий запит + очищення
    g.to_parquet(path, index=False)              #    збереження у знімок
    g = gpd.read_parquet(path)
    COST[name] = {'спосіб': 'живий запит', 'записів': len(g),   # 3) вартість
                  'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                  'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                  'секунд': round(time.time() - t0, 1)}
    return g


In [7]:
boundary = fetch('boundary', lambda: ox.geocode_to_gdf(CITY))
poly = boundary.geometry.iloc[0]
AREA_KM2 = boundary.to_crs(METRIC_CRS).area.iloc[0] / 1e6
print('межа міста отримана, площа %.1f км²' % AREA_KM2)
assert 20 < AREA_KM2 < 1500, ('площа межі %.0f км² — це не місто (область чи район?). '
                              'Отримайте контур самого міста.' % AREA_KM2)

межа міста отримана, площа 83.8 км²


In [8]:
heritage = fetch('heritage', lambda: ox.features_from_polygon(
    poly, {'historic': True, 'tourism': TOURISM_OK}))
stations = fetch('stations', lambda: ox.features_from_polygon(
    poly, {'railway': ['station', 'halt'], 'building': ['train_station']}))
amenities = fetch('amenities', lambda: ox.features_from_polygon(
    poly, {'amenity': ['cafe', 'restaurant', 'drinking_water', 'toilets']}))

# Мережа приходить двома таблицями одразу, тому їй окрема гілка.
NODES_P = os.path.join(OUT, 'road_nodes.parquet')
EDGES_P = os.path.join(OUT, 'road_edges.parquet')

if os.path.exists(NODES_P) and os.path.exists(EDGES_P):
    t0 = time.time()
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'знімок', 'записів': len(nodes) + len(edges),
                           'передано_МБ': 0.0,
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 3)}
else:
    t0, b0 = time.time(), cache_bytes()
    G_raw = ox.graph_from_polygon(poly, network_type='walk')
    nodes, edges = ox.convert.graph_to_gdfs(G_raw)
    clean(nodes).to_parquet(NODES_P, index=False)
    clean(edges).to_parquet(EDGES_P, index=False)
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'живий запит', 'записів': len(nodes) + len(edges),
                           'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 1)}

print('вузлів %d | ребер %d' % (len(nodes), len(edges)))

вузлів 8449 | ребер 22858


### Завдання 1.2

Зведіть `COST` у таблицю і дайте відповідь у звіті: **у скільки разів
читання знімка швидше за живий запит** і **скільки мегабайтів ви не передали
мережею** при повторному запуску.

Контрольна ознака правильності: при **другому** запуску ноутбука всі рядки
мають мати спосіб «знімок», а `передано_МБ` — нуль.

In [9]:
display(pd.DataFrame(COST).T)

,спосіб,записів,передано_МБ,файл_МБ,секунд
boundary,знімок,1,0.0,0.021,0.074
heritage,знімок,121,0.0,0.082,0.049
stations,знімок,9,0.0,0.023,0.028
amenities,знімок,277,0.0,0.09,0.048
мережа walk,знімок,31307,0.0,1.423,0.22


> **Чому пішохідна мережа, а не автомобільна.** У старому місті музеї стоять
> за сотню метрів один від одного, а автомобільних вузлів там одиниці. Кілька
> пам'яток сядуть на **один** вузол графа, і матриця віддалей скаже, що між
> ними нуль метрів. Задача розв'яжеться, відповідь буде неправильною, і
> повідомлення про помилку не з'явиться. У кроці 4 ви це перевірите явно.

---
## 2. Хто претендує на звання пам'ятки

У OpenStreetMap немає тега «визначна пам'ятка». Є два сімейства:

* `historic=*` — `castle`, `church`, `monument`, `memorial`, `ruins`, `tower`, `yes`…
* `tourism=*` — `museum`, `gallery`, `attraction`, `artwork`, `viewpoint`

Перетин неповний: музей у сучасній будівлі має лише `tourism`, руїни фортеці —
лише `historic`. Тому беремо **об'єднання**.

📖 <https://wiki.openstreetmap.org/wiki/Map_Features>

### Завдання 2.1

Відберіть кандидатів і **назвіть у звіті, скільки об'єктів ви відкинули й чому**.

In [ ]:
mask = heritage['historic'].notna() | heritage['tourism'].isin(TOURISM_OK)   # ОБ'ЄДНАННЯ, не перетин
has_name = heritage['name'].notna() & (heritage['name'].astype(str).str.strip() != '')
cand = heritage[mask & has_name].reset_index(drop=True)
cand['osm_key'] = cand['element'].astype(str) + '/' + cand['id'].astype(str)

print('усього у шарі heritage :', len(heritage))
print('підходять за тегами    :', int(mask.sum()))
print('з них мають name       :', len(cand))
print('відкинуто (немає name) : %d (%.1f%%)' % ((mask & ~has_name).sum(), 100 * (mask & ~has_name).sum() / mask.sum()))
print()
print('що саме відкинуто: найчастіші historic серед безіменних:')
print(heritage[mask & ~has_name]['historic'].value_counts().head(6).to_string())

усього у шарі heritage : 121
підходять за тегами    : 121
з них мають name       : 82
відкинуто (немає name) : 39 (32.2%)

що саме відкинуто — найчастіші historic серед безіменних:
historic
cannon      12
memorial    11
ruins        5
building     2
wreck        1


In [11]:
assert 'cand' in dir(), 'спершу виконайте завдання 2.1'
print('=== historic ===');  print(cand['historic'].value_counts().head(10).to_string())
print()
print('=== tourism ===');   print(cand['tourism'].value_counts().head(10).to_string())
print()
print('типи геометрії:', cand.geometry.geom_type.value_counts().to_dict())

=== historic ===
historic
memorial               31
aircraft                9
cannon                  5
tomb                    4
tank                    2
monument                2
yes                     2
archaeological_site     1
vehicle                 1
synagogue               1

=== tourism ===
tourism
museum        8
artwork       7
attraction    4
viewpoint     1

типи геометрії: {'Point': 64, 'Polygon': 18}


> **Перевірте себе.** Серед типів геометрії мають бути і `Point`, і `Polygon`.
> Якщо самі лише полігони — ви відфільтрували зайве: пам'ятники, меморіали та
> оглядові точки в OSM є **вузлами**, у них немає будівлі.

### Якщо кандидатів менше тридцяти

Невелике місто може дати 15–20 об'єктів. Тоді:

1. розширте перелік `tourism` (наприклад, додайте `theatre`, `place_of_worship`);
2. або зменште `N_LARGE` до того, що є;

і **обов'язково зазначте це у звіті** разом із числом.

---
## 3. Критерій «найкращі»

Ключовий крок роботи. Стовпця «цінність» у даних немає. Його треба
сконструювати і захистити.

Сигнали, які є в OSM:

| сигнал | що означає |
|---|---|
| `wikidata` / `wikipedia` | про об'єкт є стаття — хтось сторонній визнав його вартим опису |
| кількість колонок `name:*` | назву переклали N мовами — перекладають те, що шукають іноземці |
| `tourism=museum/gallery` | призначений для відвідування |
| `historic` заповнений | має історичну природу |
| площа полігона | розмір — **найслабший сигнал**: великим буває і склад |

### Завдання 3.1

Побудуйте числовий рейтинг. Вимоги:

* використати **щонайменше три** різні сигнали;
* звести різнорідні величини до спільної шкали (підказка: `rank(pct=True)`);
* **пояснити у звіті, чому саме такі ваги**.

In [12]:
import re
metric = cand.to_crs(METRIC_CRS)
cand['area_m2'] = np.where(metric.geometry.geom_type.isin(['Polygon', 'MultiPolygon']),
                           metric.area.values, 0.0)

# Сигнали ----------------------------------------------------------------
LANG_COLS = [c for c in cand.columns if re.fullmatch(r'name:[a-z]{2,3}', c) and c != 'name:uk']   # мовні name:xx, крім рідної української
print('мовні колонки name:* :', LANG_COLS)

cand['wiki']   = (cand['wikidata'].notna() | cand['wikipedia'].notna()).astype(int)   # зовнішнє визнання
cand['n_lang'] = cand[LANG_COLS].notna().sum(axis=1)                                   # скільки мов
cand['visit']  = cand['tourism'].isin(['museum', 'gallery', 'attraction']).astype(int) # призначений для відвідування
cand['hist']   = cand['historic'].notna().astype(int)                                  # має історичну природу

# Формула: різнорідні величини зведено до шкали 0..1.
# wiki, visit, hist — вже 0/1; n_lang — частка від 0 до 1 через rank(pct=True).
# method='min': всі об'єкти без жодного перекладу отримують найнижчий ранг (≈0), а не середній.
W = {'wiki': 0.40, 'n_lang': 0.30, 'visit': 0.20, 'hist': 0.10}
cand['score'] = (W['wiki']   * cand['wiki']
               + W['n_lang'] * cand['n_lang'].rank(pct=True, method='min')
               + W['visit']  * cand['visit']
               + W['hist']   * cand['hist'])

print('кандидатів   :', len(cand))
print('мають wiki   :', int(cand['wiki'].sum()))
print('мають ≥1 мови:', int((cand['n_lang'] > 0).sum()))
print('ваги         :', W)

мовні колонки name:* : ['name:pl', 'name:en', 'name:ru']
кандидатів   : 82
мають wiki   : 23
мають ≥1 мови: 17
ваги         : {'wiki': 0.4, 'n_lang': 0.3, 'visit': 0.2, 'hist': 0.1}


### Завдання 3.2 — обов'язкове порівняння

Побудуйте **другий** рейтинг за найпростішим критерієм — площею будівлі —
і порахуйте, скільки об'єктів збігається в перших десяти.

Це головне питання роботи: обидва списки отримані з тих самих даних, тим
самим кодом, без жодної помилки. Якщо вони різні — поясніть у звіті, що це
означає для слова «найкращі».

In [13]:
N_SMALL, N_LARGE = 10, 30

def rank_by(df, col):
    # ничия за рахунком вирішує назва, щоб результат був відтворюваним
    return df.sort_values([col, 'name'], ascending=[False, True], kind='mergesort')

top_score = rank_by(cand, 'score').head(N_LARGE).reset_index(drop=True)
top10     = top_score.head(N_SMALL)
top_naive = rank_by(cand, 'area_m2').head(N_SMALL).reset_index(drop=True)
overlap   = len(set(top10['osm_key']) & set(top_naive['osm_key']))
overlap_raw = overlap                       # до чищення дублікатів
print('перетин: %d з %d' % (overlap, N_SMALL))
display(pd.DataFrame({'за вашим критерієм': list(top10['name']),
                      'за площею': list(top_naive['name'])}))

# чутливість до ваг: якщо всі чотири сигнали рівні, наскільки зміниться десятка?
eq = (0.25 * cand['wiki'] + 0.25 * cand['n_lang'].rank(pct=True, method='min')
      + 0.25 * cand['visit'] + 0.25 * cand['hist'])
top10_eq = set(rank_by(cand.assign(eq=eq), 'eq').head(N_SMALL)['osm_key'])
overlap_eq = len(top10_eq & set(top10['osm_key']))
print('перетин із рівновагомим варіантом: %d з %d' % (overlap_eq, N_SMALL))

перетин: 2 з 10


,за вашим критерієм,за площею
0,Антонієві печери,Національний архітектурно-історичний заповідни...
1,Художній музей,Болдина гора
2,Дім полкової канцелярії,Літературно-меморіальний музей-заповідник Миха...
3,Літературно-меморіальний музей-заповідник Миха...,Художній музей
4,Козацький кафедральний собор Святої Великомуче...,Нудистський пляж
5,Івану Мазепі,АТБ-Маркет
6,Битва за Чернігів,Спасо-Преображенський собор
7,Богдану Хмельницькому,Введенська церква (трапезна)
8,Т.Г. Шевченко,Чернігівський Колегіум
9,Чорна могила,Молодіжний театр


перетин із рівновагомим варіантом: 10 з 10


### Завдання 3.3 — дублікати

OSM наповнюють різні люди. Той самий музей трапляється двічі під різними
назвами, а вежа буває окремим об'єктом усередині палацу. Знайдіть у `top_score`
пари, що стоять ближче **60 метрів** одна до одної, і вирішіть, що з ними робити.

Рішення (лишити / злити / видалити) опишіть у звіті.

In [ ]:
def close_pairs(gdf, limit_m=60):
    p = gdf.to_crs(METRIC_CRS).geometry.representative_point()
    XY = np.c_[p.x.values, p.y.values]
    D = np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(D, np.inf)
    out = [(i, j, D[i, j]) for i in range(len(XY)) for j in range(i + 1, len(XY))
           if D[i, j] < limit_m]
    return sorted(out, key=lambda t: t[2])

pairs = close_pairs(top_score, 60)
print('пар ближче 60 м у топ-%d (до чищення): %d' % (N_LARGE, len(pairs)))
for i, j, d in pairs:
    print('  %5.1f м | %s  <->  %s' % (d, top_score['name'][i], top_score['name'][j]))


lines = cand[cand.geometry.geom_type == 'LineString']
print('лінійних об’єктів виключено з маршруту: %d' % len(lines))
for nm in lines['name'].head(8):
    print('   ', nm)

dropped_d, dropped_same = {}, []
cand_nd = cand[cand.geometry.geom_type != 'LineString'].copy()
while True:
    cur = rank_by(cand_nd, 'score').head(N_LARGE).reset_index(drop=True)
    pr = close_pairs(cur, 60)
    same = cur[cur.duplicated('name', keep='first')]
    if not pr and same.empty:
        break
    for i, j, d in pr:
        dropped_d.setdefault(cur['osm_key'][j], (cur['name'][j], cur['name'][i], round(d, 1)))
    for _, r in same.iterrows():
        dropped_same.append(r['name'])
    kill = {cur['osm_key'][j] for i, j, d in pr} | set(same['osm_key'])
    cand_nd = cand_nd[~cand_nd['osm_key'].isin(kill)]

dropped = list(dropped_d.values())
top_score = cur
top10 = top_score.head(N_SMALL)
print('\nвидалено як дублікати: %d' % len(dropped))
for nm, keep, d in dropped:
    print('  видалено «%s» (%.1f м від «%s»)' % (nm, d, keep))
print('видалено як однойменні копії: %d' % len(dropped_same), sorted(set(dropped_same)))
overlap = len(set(top10['osm_key']) & set(top_naive['osm_key']))
print('\nперетин після чищення: %d з %d' % (overlap, N_SMALL))
print('\nтоп-%d після чищення:' % N_LARGE)
display(top_score[['name', 'historic', 'tourism', 'wiki', 'n_lang', 'score']].round(3))

пар ближче 60 м у топ-30 (до чищення): 5
    7.2 м | Чорна могила  <->  Чорна могила
   23.1 м | Спасо-Преображенський собор  <->  Національний архітектурно-історичний заповідник "Чернігів стародавній"
   32.8 м | Івану Мазепі  <->  Чернігівський Колегіум
   48.6 м | Чернігівський Колегіум  <->  Борисоглібський собор
   53.6 м | Борисоглібський собор  <->  Національний архітектурно-історичний заповідник "Чернігів стародавній"
лінійних об’єктів виключено з маршруту: 0

видалено як дублікати: 5
  видалено «Чорна могила» (7.2 м від «Чорна могила»)
  видалено «Національний архітектурно-історичний заповідник "Чернігів стародавній"» (23.1 м від «Спасо-Преображенський собор»)
  видалено «Чернігівський Колегіум» (32.8 м від «Івану Мазепі»)
  видалено «Борисоглібський собор» (48.6 м від «Чернігівський Колегіум»)
  видалено «ІСУ-152» (38.9 м від «Військово-історичний музей»)
видалено як однойменні копії: 1 ['Чорна могила']

перетин після чищення: 2 з 10

топ-30 після чищення:


,name,historic,tourism,wiki,n_lang,score
0,Антонієві печери,<NA>,attraction,1,2,0.889
1,Художній музей,<NA>,museum,1,2,0.889
2,Дім полкової канцелярії,<NA>,attraction,1,1,0.841
3,Літературно-меморіальний музей-заповідник Миха...,<NA>,museum,1,1,0.841
4,Козацький кафедральний собор Святої Великомуче...,church,<NA>,1,2,0.789
5,Івану Мазепі,monument,<NA>,1,1,0.741
6,Битва за Чернігів,yes,<NA>,1,1,0.741
7,Богдану Хмельницькому,memorial,<NA>,1,1,0.741
8,Т.Г. Шевченко,memorial,artwork,1,1,0.741
9,Чорна могила,memorial,<NA>,1,1,0.741


---
## 4. Старт: вокзал, і прив'язка точок до графа

In [15]:
print('станції у шарі:', len(stations))
print(stations[['name', 'railway', 'building']].dropna(how='all').head(12).to_string())

станції у шарі: 9
                              name  railway       building
0                         Чернігів  station           <NA>
1                        Подусівка     halt           <NA>
2               Чернігів-Північний  station           <NA>
3               Залізничний вокзал     <NA>  train_station
4     Автовокзал "Чернігів" (АС-1)     <NA>            yes
5           Автостанція Чернігів 2     <NA>            yes
6                             <NA>     <NA>  train_station
7  Автостанція "Чернігів-2" (АС-2)     <NA>           <NA>
8  Автостанція "Чернігів-3" (АС-3)     <NA>           <NA>


In [16]:
CITY_NAME = CITY.split(',')[0].strip()
station = stations[(stations['railway'] == 'station')
                  & (stations['name'] == CITY_NAME)].reset_index(drop=True)   # станція з назвою міста
assert len(station) == 1, 'очікується рівно один головний вокзал, знайдено %d' % len(station)
print('старт:', station['name'].iloc[0], '|', station.geometry.iloc[0].geom_type)
MAP_CENTER = [station.geometry.iloc[0].y, station.geometry.iloc[0].x]

старт: Чернігів | Point


In [17]:
G = nx.DiGraph()
for t in nodes.itertuples():
    G.add_node(t.osmid, x=t.x, y=t.y)
for t in edges.itertuples():
    L = float(t.length)
    if G.has_edge(t.u, t.v):
        G[t.u][t.v]['length'] = min(G[t.u][t.v]['length'], L)
    else:
        G.add_edge(t.u, t.v, length=L)
print('граф: %d вузлів, %d ребер' % (G.number_of_nodes(), G.number_of_edges()))

NODES_M = nodes.to_crs(METRIC_CRS)
NODE_XY = np.c_[NODES_M.geometry.x.values, NODES_M.geometry.y.values]
NODE_ID = nodes['osmid'].values

def snap(points_gdf):
    """Найближчий вузол графа до кожної точки. Рахуємо в метричній CRS."""
    pmet = points_gdf.to_crs(METRIC_CRS)
    ids, dist = [], []
    for p in pmet.geometry:
        d2 = (NODE_XY[:, 0] - p.x) ** 2 + (NODE_XY[:, 1] - p.y) ** 2
        k = int(np.argmin(d2))
        ids.append(NODE_ID[k]); dist.append(float(np.sqrt(d2[k])))
    return np.array(ids), np.array(dist)

def make_points(top):
    names = ['СТАРТ: ' + str(station['name'].iloc[0])] + list(top['name'])
    geoms = [station.geometry.iloc[0].representative_point()] + \
            [g.representative_point() for g in top.geometry]
    return gpd.GeoDataFrame({'name': names}, geometry=geoms, crs='EPSG:4326')

граф: 8449 вузлів, 22798 ребер


### Завдання 4.1 — обов'язкова перевірка

Напишіть `check_snap`, яка повідомляє:

* скільки точок і скільки **різних** вузлів вони зайняли;
* медіану й максимум відстані «точка → вузол»;
* якщо вузлів менше, ніж точок — **назви точок, що злиплись**.

**Контрольна ознака:** на пішохідній мережі різних вузлів має бути **стільки ж,
скільки точок**. Якщо менше — ви взяли автомобільну мережу або місто надто мале;
зафіксуйте це у звіті, бо далі такі пари отримають нульову віддаль.

In [18]:
def check_snap(points, ids, dist, label):
    n, k = len(points), len(set(ids))
    print('%s: точок %d, різних вузлів %d | відстань точка→вузол: медіана %.1f м, максимум %.1f м'
          % (label, n, k, float(np.median(dist)), float(np.max(dist))))
    if k < n:
        for node, c in collections.Counter(ids).items():
            if c > 1:
                who = [points['name'].iloc[i] for i in range(n) if ids[i] == node]
                print('   злиплись на вузлі %s: %s' % (node, ' | '.join(who)))
    else:
        print('   ✓ кожна точка має власний вузол')
    return k == n

---
## 5. Матриця віддалей

Дві матриці: **пряма лінія** і **дорогою**. Їхнє відношення — коефіцієнт
обходу — показує, наскільки пряма лінія бреше.

### Завдання 5.1

In [19]:
def road_matrix(ids):
    """Матриця найкоротших віддалей графом між заданими вузлами."""
    k = len(ids)
    D = np.full((k, k), np.inf)
    for i, s in enumerate(ids):
        lengths = nx.single_source_dijkstra_path_length(G, s, weight='length')
        for j, t in enumerate(ids):
            D[i, j] = lengths.get(t, np.inf)
    np.fill_diagonal(D, 0.0)
    return D

def straight_matrix(points):
    pm = points.to_crs(METRIC_CRS)                       # метри, не градуси
    XY = np.c_[pm.geometry.x.values, pm.geometry.y.values]
    return np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))

def detour_factor(D, E):
    ok = np.isfinite(D) & (E > 0)
    np.fill_diagonal(ok, False)
    return float(np.median(D[ok] / E[ok]))


**Контрольні ознаки — звіртесь, не підганяючи:**

* коефіцієнт обходу для пішохідної мережі у місті зазвичай **1,2–1,6**;
* якщо у вас рівно **1,0** — ви порівнюєте матрицю саму із собою;
* якщо **більше 2** — найімовірніше взято автомобільну мережу з односторонніми
  вулицями, або місто має річку/залізницю, яку нема де перетнути. Друге —
  цікавий результат, перше — помилка. Розберіться, що саме у вас, і напишіть це.

Якщо серед віддалей трапились `inf` — частина графа не зв'язана з рештою.
Не замінюйте `inf` великим числом мовчки: спершу з'ясуйте, які це точки.

In [20]:
pts10 = make_points(top10)
ids10, dist10 = snap(pts10)
check_snap(pts10, ids10, dist10, 'мала задача')

t0 = time.time(); D10 = road_matrix(ids10); print('матриця за %.3f с' % (time.time() - t0))
E10 = straight_matrix(pts10)
print('недосяжних пар    :', int(np.isinf(D10).sum()))
print('асиметрія         : %.0f м' % float(np.max(np.abs(D10 - D10.T))))
print('коефіцієнт обходу : %.2f' % detour_factor(D10, E10))

мала задача: точок 11, різних вузлів 11 | відстань точка→вузол: медіана 17.2 м, максимум 76.1 м
   ✓ кожна точка має власний вузол


матриця за 0.199 с


недосяжних пар    : 0
асиметрія         : 0 м
коефіцієнт обходу : 1.20


> **Асиметрія.** $D_{ij} \\neq D_{ji}$ — не помилка: односторонній рух робить
> шлях туди і назад різним. На пішохідній мережі асиметрія мала, на
> автомобільній — помітна. Подивіться на своє число і поясніть його у звіті.

---
## 6. Задача комівояжера

| метод | що робить | складність |
|---|---|---|
| найближчий сусід | щоразу йде в найближчу невідвідану точку | $O(n^2)$ |
| 2-opt | розвертає ділянки готового маршруту, поки коротшає | $O(n^2)$ за прохід |
| Held–Karp | динамічне програмування за підмножинами — **точний** | $O(2^n n^2)$ часу, $O(2^n n)$ пам'яті |

Held–Karp дано готовим: він не є метою роботи, але без нього ви не дізнаєтесь,
**наскільки** помиляються перші два.

### Завдання 6.1 — реалізуйте перші два методи

In [21]:
def tour_length(D, tour):
    return float(sum(D[tour[i], tour[i + 1]] for i in range(len(tour) - 1)))

def nearest_neighbour(D):
    """Маршрут 0 -> ... -> 0. Щоразу йдемо в найближчу невідвідану точку."""
    k = len(D); tour = [0]; left = set(range(1, k))
    while left:
        cur = tour[-1]
        nxt = min(left, key=lambda j: D[cur, j])
        tour.append(nxt); left.remove(nxt)
    return tour + [0]

def two_opt(D, tour):
    """Беремо маршрут і розвертаємо ділянки, поки довжина зменшується."""
    tour = list(tour); best = tour_length(D, tour); improved = True
    while improved:
        improved = False
        for i in range(1, len(tour) - 2):
            for j in range(i + 1, len(tour) - 1):
                new = tour[:i] + tour[i:j + 1][::-1] + tour[j + 1:]
                L = tour_length(D, new)        # повний перерахунок: коректно і для несиметричної D
                if L < best - 1e-9:
                    tour, best, improved = new, L, True
    return tour


In [22]:
def held_karp(D):
    """Точний розв'язок. Вузол 0 — старт і фініш. ДАНО, писати не треба."""
    k = len(D); N = k - 1; INF = float('inf')
    dp = [[INF] * N for _ in range(1 << N)]
    par = [[-1] * N for _ in range(1 << N)]
    for j in range(N):
        dp[1 << j][j] = D[0, j + 1]
    for S in range(1 << N):
        for j in range(N):
            cur = dp[S][j]
            if cur == INF or not (S >> j) & 1:
                continue
            for t in range(N):
                if (S >> t) & 1:
                    continue
                S2 = S | (1 << t); v = cur + D[j + 1, t + 1]
                if v < dp[S2][t]:
                    dp[S2][t] = v; par[S2][t] = j
    full = (1 << N) - 1
    bj = min(range(N), key=lambda j: dp[full][j] + D[j + 1, 0])
    best = dp[full][bj] + D[bj + 1, 0]
    path = []; S = full; j = bj
    while j != -1:
        path.append(j + 1); pj = par[S][j]; S ^= (1 << j); j = pj
    return [0] + path[::-1] + [0], float(best)

### Самоперевірка

Ці твердження мають виконатись для **будь-якого** міста. Якщо хоч одне впало —
шукайте помилку в коді, а не в даних.

In [23]:
def check_tour(tour, k, label):
    assert tour[0] == 0 and tour[-1] == 0, '%s: маршрут має починатись і закінчуватись на старті' % label
    assert sorted(tour[:-1]) == list(range(k)), '%s: кожна точка рівно один раз' % label
    print('%s: структура маршруту правильна (%d точок)' % (label, k))

nn10 = nearest_neighbour(D10)
op10 = two_opt(D10, nn10)
ex10, L_opt = held_karp(D10)

for t, lbl in [(nn10, 'найближчий сусід'), (op10, '2-opt'), (ex10, 'Held-Karp')]:
    check_tour(t, len(ids10), lbl)

assert tour_length(D10, op10) <= tour_length(D10, nn10) + 1e-6, '2-opt не може бути гіршим за вхідний маршрут'
assert L_opt <= tour_length(D10, op10) + 1e-6, 'точний розв\u2019язок не може бути гіршим за наближений'
print()
print('усі перевірки пройдено')

найближчий сусід: структура маршруту правильна (11 точок)
2-opt: структура маршруту правильна (11 точок)
Held-Karp: структура маршруту правильна (11 точок)

усі перевірки пройдено


In [24]:
rows = []
for name, tour in [('найближчий сусід', nn10), ('2-opt', op10), ('Held-Karp (точний)', ex10)]:
    L = tour_length(D10, tour)
    rows.append({'метод': name, 'довжина, м': round(L),
                 'гірше оптимуму, %': round(100 * (L / L_opt - 1), 2)})
display(pd.DataFrame(rows).set_index('метод'))

print()
print('оптимальний порядок:')
for step, i in enumerate(ex10):
    print('  %2d. %s' % (step, pts10['name'].iloc[i]))
print()
print('довжина: %.2f км' % (L_opt / 1000))

,"довжина, м","гірше оптимуму, %"
метод,,
найближчий сусід,11837,7.83
2-opt,11108,1.19
Held-Karp (точний),10977,0.00



оптимальний порядок:
   0. СТАРТ: Чернігів
   1. Літературно-меморіальний музей-заповідник Михайла Коцюбинського
   2. Битва за Чернігів
   3. Богдану Хмельницькому
   4. Івану Мазепі
   5. Художній музей
   6. Дім полкової канцелярії
   7. Т.Г. Шевченко
   8. Козацький кафедральний собор Святої Великомучениці Катерини
   9. Чорна могила
  10. Антонієві печери
  11. СТАРТ: Чернігів

довжина: 10.98 км


### Завдання 6.2 — велика задача

Спершу оцініть, чи можливий тут точний розв'язок. Порахуйте для
$n = 11, 16, 21, 26, 31$ розмір таблиці Held–Karp ($2^{n-1}\\cdot(n-1)$ клітин)
і скільки це гігабайтів при 8 байтах на клітину.

In [25]:
def human(b):
    for u in ['Б', 'КБ', 'МБ', 'ГБ', 'ТБ']:
        if b < 1000: return '%.3g %s' % (b, u)
        b /= 1000
    return '%.3g ПБ' % b

rows = []
for n in [11, 16, 21, 26, 31]:
    cells = 2 ** (n - 1) * (n - 1)
    rows.append({'точок': n, 'клітин у таблиці': '%.1e' % cells,
                 'пам\u2019яті': human(cells * 8),
                 'операцій': '%.1e' % (2 ** (n - 1) * (n - 1) ** 2)})
display(pd.DataFrame(rows).set_index('точок'))

,клітин у таблиці,пам’яті,операцій
точок,,,
11,1.0e+04,81.9 КБ,1.0e+05
16,4.9e+05,3.93 МБ,7.4e+06
21,2.1e+07,168 МБ,4.2e+08
26,8.4e+08,6.71 ГБ,2.1e+10
31,3.2e+10,258 ГБ,9.7e+11


Подивіться на останній рядок і напишіть у звіті, **чому саме** точний розв'язок
на 30 точках недоступний. Слово «довго» відповіддю не є.

In [26]:
pts30 = make_points(top_score)
ids30, dist30 = snap(pts30)
check_snap(pts30, ids30, dist30, 'велика задача')

D30 = road_matrix(ids30)
E30 = straight_matrix(pts30)
print('коефіцієнт обходу: %.2f' % detour_factor(D30, E30))

nn30 = nearest_neighbour(D30)
t0 = time.time(); op30 = two_opt(D30, nn30); t_op30 = time.time() - t0
check_tour(op30, len(ids30), '2-opt на 30 точках')

L_nn30, L_op30 = tour_length(D30, nn30), tour_length(D30, op30)
print()
print('найближчий сусід : %8.0f м' % L_nn30)
print('2-opt            : %8.0f м  за %.3f с  (коротше на %.1f%%)'
      % (L_op30, t_op30, 100 * (1 - L_op30 / L_nn30)))
print()
print('ЦЕ — база порівняння для лабораторної роботи № 8.')

велика задача: точок 31, різних вузлів 31 | відстань точка→вузол: медіана 22.9 м, максимум 323.9 м
   ✓ кожна точка має власний вузол


коефіцієнт обходу: 1.19
2-opt на 30 точках: структура маршруту правильна (31 точок)

найближчий сусід :    40401 м
2-opt            :    36688 м  за 0.015 с  (коротше на 9.2%)

ЦЕ — база порівняння для лабораторної роботи № 8.


---
## 7. Просторова індексація: H3

H3 — ієрархічна сітка шестикутників. Рівень 9 — приблизно 200 м по ребру.

📖 <https://h3geo.org/docs/core-library/restable/>

### Завдання 7.1

Розкладіть **усіх** кандидатів по комірках H3 рівня 9 і побудуйте картограму
щільності. На тій самій карті покажіть межу міста і точки вашого маршруту.

In [27]:
import h3
from shapely.geometry import Polygon

H3_RES = 9
print('середнє ребро комірки: %.0f м' % h3.average_hexagon_edge_length(H3_RES, unit='m'))

cpts = cand.geometry.representative_point()
cand['h3'] = [h3.latlng_to_cell(p.y, p.x, H3_RES) for p in cpts]          # (lat, lng)!
dens = cand.groupby('h3').size().rename('n').reset_index()
hexes = gpd.GeoDataFrame(
    dens,
    geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in dens['h3']],  # у shapely (x=lng, y=lat)
    crs='EPSG:4326')

srt = dens['n'].sort_values(ascending=False).reset_index(drop=True)
half = int((srt.cumsum() < len(cand) / 2).sum()) + 1
print('кандидатів: %d | комірок H3: %d | максимум в одній комірці: %d' % (len(cand), len(dens), srt.iloc[0]))
print('половина всіх пам\u2019яток (%d) зосереджена у %d комірках (%.1f%% від усіх)'
      % (len(cand) / 2, half, 100 * half / len(dens)))

import matplotlib
cmap = matplotlib.colormaps['YlOrRd']; vmax = float(hexes['n'].max())
h3map = folium.Map(location=MAP_CENTER, zoom_start=12, tiles=None)
folium.TileLayer(tiles=BASEMAP.build_url(), attr=BASEMAP.html_attribution
                 + ' | Дані © OpenStreetMap contributors, ODbL', name='підкладка').add_to(h3map)
folium.GeoJson(hexes.to_json(), name='щільність H3-9',
               style_function=lambda f: {'fillColor': matplotlib.colors.to_hex(cmap(f['properties']['n'] / vmax)),
                                         'color': '#555', 'weight': 0.4, 'fillOpacity': 0.65},
               tooltip=folium.GeoJsonTooltip(fields=['n'], aliases=['пам\u2019яток:'])).add_to(h3map)
folium.GeoJson(boundary.to_json(), name='межа міста',
               style_function=lambda f: {'fill': False, 'color': '#1b5e20', 'weight': 2}).add_to(h3map)
for i, r in enumerate(pts10.itertuples()):
    folium.CircleMarker([r.geometry.y, r.geometry.x], radius=5, color='#0d47a1', fill=True,
                        fill_opacity=1, tooltip=str(r.name)).add_to(h3map)
h3map

середнє ребро комірки: 201 м
кандидатів: 82 | комірок H3: 42 | максимум в одній комірці: 10
половина всіх пам’яток (41) зосереджена у 7 комірках (16.7% від усіх)


> **Пастка.** `h3.cell_to_boundary` повертає пари **(lat, lng)**, а
> `shapely.Polygon` чекає **(x, y)**, тобто (lng, lat). Переплутаєте — місто
> опиниться в Індійському океані. Це видно одразу, тому дивіться на карту.

### Питання до звіту

У скількох комірках зосереджено половину всіх пам'яток вашого міста?
Що це говорить про форму маршруту?

---
## 8. Просторове поєднання

### Завдання 8.1

Для кожної точки маршруту знайдіть найближчий заклад із шару `amenities`
(`sjoin_nearest`) і виведіть таблицю з відстанню в метрах.

Нагадування: `sjoin_nearest` рахує в одиницях CRS. У градусах результат буде
безглуздим — перепроєктуйте обидва шари в метричну систему.

In [28]:
am = amenities.to_crs(METRIC_CRS).copy()                      # ОБИДВА шари — у метри
am['geometry'] = am.geometry.representative_point()      # полігони -> точки
am = am[['amenity', 'name', 'geometry']].rename(columns={'name': 'amenity_name'})
rp = pts10.to_crs(METRIC_CRS)[['name', 'geometry']]
assert am.crs == rp.crs and am.crs.is_projected

near = gpd.sjoin_nearest(rp, am, how='left', distance_col='dist_m')
near = near[~near.index.duplicated(keep='first')]         # при рівних відстанях лишаємо один
display(near[['name', 'amenity', 'amenity_name', 'dist_m']].round({'dist_m': 0}).reset_index(drop=True))
print('медіана відстані до найближчого закладу: %.0f м, максимум %.0f м'
      % (near['dist_m'].median(), near['dist_m'].max()))

,name,amenity,amenity_name,dist_m
0,СТАРТ: Чернігів,toilets,<NA>,132.0
1,Антонієві печери,toilets,<NA>,49.0
2,Художній музей,toilets,<NA>,151.0
3,Дім полкової канцелярії,toilets,<NA>,159.0
4,Літературно-меморіальний музей-заповідник Миха...,drinking_water,<NA>,44.0
5,Козацький кафедральний собор Святої Великомуче...,toilets,<NA>,165.0
6,Івану Мазепі,restaurant,Хінкальня,116.0
7,Битва за Чернігів,pharmacy,Подорожник,65.0
8,Богдану Хмельницькому,cafe,Coffe Boom,107.0
9,Т.Г. Шевченко,drinking_water,<NA>,123.0


медіана відстані до найближчого закладу: 123 м, максимум 259 м


---
## 9. Маршрут на карті

Лінія маршруту — це **геометрія ребер графа**, а не відрізки між пам'ятками.

### Завдання 9.1

Побудуйте карту, на якій **обидві** лінії: справжній маршрут по ребрах і
прямі відрізки між точками. Різниця має бути видно оком.

In [29]:
node_xy = nodes.set_index('osmid')[['x', 'y']]

# геометрія ребра (u, v): найкоротше з паралельних; ребра мають і форму, а не лише кінці
EDGE_GEOM = {}
for t in edges.itertuples():
    k = (t.u, t.v)
    if k not in EDGE_GEOM or t.length < EDGE_GEOM[k][0]:
        EDGE_GEOM[k] = (t.length, t.geometry)

def route_coords(tour, ids):
    """Координати маршруту по ребрах графа (з їхньою справжньою формою), формат (lat, lon) для folium."""
    coords = []
    for a, b in zip(tour[:-1], tour[1:]):
        try:
            path = nx.shortest_path(G, ids[a], ids[b], weight='length')
        except nx.NetworkXNoPath:
            path = [ids[a], ids[b]]
        for u, v in zip(path[:-1], path[1:]):
            g = EDGE_GEOM.get((u, v))
            if g is not None:
                coords.extend((y, x) for x, y in g[1].coords)
            else:
                coords.extend([(float(node_xy.loc[u, 'y']), float(node_xy.loc[u, 'x'])),
                               (float(node_xy.loc[v, 'y']), float(node_xy.loc[v, 'x']))])
    return coords

def straight_coords(tour, pts):
    return [(pts.geometry.y.iloc[i], pts.geometry.x.iloc[i]) for i in tour]

def numbered_marker(lat, lon, text, tip, color):
    html = ('<div style="background:%s;color:#fff;border-radius:50%%;width:24px;height:24px;'
            'line-height:24px;text-align:center;font:bold 12px sans-serif;border:2px solid #fff;'
            'box-shadow:0 0 3px #000">%s</div>' % (color, text))
    return folium.Marker([lat, lon], icon=folium.DivIcon(html=html, icon_size=(24, 24), icon_anchor=(12, 12)),
                         tooltip=tip)

fmap = folium.Map(location=MAP_CENTER, zoom_start=14, tiles=None)
folium.TileLayer(tiles=BASEMAP.build_url(), attr=BASEMAP.html_attribution
                 + ' | Дані © OpenStreetMap contributors, ODbL', name='підкладка').add_to(fmap)

road10 = route_coords(ex10, ids10)
fg = folium.FeatureGroup('10 точок: Held–Karp, по ребрах графа')
folium.PolyLine(road10, color='#1565c0', weight=5, opacity=0.9).add_to(fg); fg.add_to(fmap)
fg = folium.FeatureGroup('10 точок: прямі відрізки між точками')
folium.PolyLine(straight_coords(ex10, pts10), color='#e53935', weight=3, dash_array='6,8').add_to(fg); fg.add_to(fmap)
fg = folium.FeatureGroup('10 точок: пункти за порядком обходу')
for step, i in enumerate(ex10[:-1]):
    numbered_marker(pts10.geometry.y.iloc[i], pts10.geometry.x.iloc[i],
                    'С' if step == 0 else str(step), '%d. %s' % (step, pts10['name'].iloc[i]),
                    '#2e7d32' if step == 0 else '#0d47a1').add_to(fg)
fg.add_to(fmap)

road30 = route_coords(op30, ids30)
fg = folium.FeatureGroup('30 точок: 2-opt, по ребрах (база для ЛР8)', show=False)
folium.PolyLine(road30, color='#6a1b9a', weight=4, opacity=0.8).add_to(fg)
for step, i in enumerate(op30[:-1]):
    folium.CircleMarker([pts30.geometry.y.iloc[i], pts30.geometry.x.iloc[i]], radius=4, color='#6a1b9a',
                        fill=True, fill_opacity=1, tooltip='%d. %s' % (step, pts30['name'].iloc[i])).add_to(fg)
fg.add_to(fmap)
folium.LayerControl(collapsed=False).add_to(fmap)
fmap.save('lr5_маршрут.html')

L_road10 = sum(EDGE_GEOM[(u, v)][0] for a, b in zip(ex10[:-1], ex10[1:])
               for u, v in zip(nx.shortest_path(G, ids10[a], ids10[b], weight='length')[:-1],
                               nx.shortest_path(G, ids10[a], ids10[b], weight='length')[1:]))
L_straight10 = float(sum(E10[a, b] for a, b in zip(ex10[:-1], ex10[1:])))
print('маршрут по ребрах: %.2f км | сума прямих відрізків: %.2f км | прямі коротші в %.2f раза'
      % (L_road10 / 1000, L_straight10 / 1000, L_road10 / L_straight10))
print('збережено lr5_маршрут.html')
fmap

маршрут по ребрах: 10.98 км | сума прямих відрізків: 8.07 км | прямі коротші в 1.36 раза
збережено lr5_маршрут.html


---
## 10. Результати для лабораторної роботи № 8

Збережіть файл, який ЛР8 прочитає і спробує побити.

In [30]:
result = {
    'студент': STUDENT_NAME,
    'номер': LIST_NUMBER,
    'місто': CITY,
    'мала_задача': {
        'точок': len(ids10),
        'оптимум_м': round(L_opt, 1),
        'оптимальний_порядок': [str(pts10['name'].iloc[i]) for i in ex10],
        'найближчий_сусід_м': round(tour_length(D10, nn10), 1),
        'двоопт_м': round(tour_length(D10, op10), 1),
    },
    'велика_задача': {
        'точок': len(ids30),
        'найближчий_сусід_м': round(L_nn30, 1),
        'двоопт_м': round(L_op30, 1),
        'двоопт_с': round(t_op30, 4),
        'точки': [{'name': str(pts30['name'].iloc[i]),
                   'lat': float(pts30.geometry.y.iloc[i]),
                   'lon': float(pts30.geometry.x.iloc[i])} for i in range(len(pts30))],
        'матриця_віддалей_м': [[None if not np.isfinite(v) else round(float(v), 1)
                                for v in row] for row in D30],
    },
}
with open('lr5_результати.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=1)
print('збережено lr5_результати.json (%.1f КБ)' % (os.path.getsize('lr5_результати.json') / 1024))

збережено lr5_результати.json (16.1 КБ)


---
## 11. Звіт

Відповіді пишіть просто в наступній комірці, числами зі своїх обчислень.

1. **Скільки об'єктів ви відкинули** на кроці 2 і за яким правилом? Чи погодились
   би ви віддати цей фільтр замовнику без пояснень?
2. **Які сигнали ввійшли у ваш рейтинг і з якими вагами?** Чому саме так?
3. **Скільки об'єктів збіглося** у ваших двох рейтингах із десяти? Що це
   означає для слова «найкращі»?
4. **Який у вас коефіцієнт обходу** і чи типовий він? Якщо ні — що в місті
   його пояснює?
5. **На скільки відсотків** найближчий сусід гірший за точний розв'язок?
   Чи здивувало вас це число?
6. **Чому точний розв'язок на 30 точках недоступний?** Назвіть конкретну
   величину, яка цьому заважає.
7. **Що у ваших даних виявилось неправильним** — дублікати, пропущені назви,
   об'єкти не на місці? Як ви з цим вчинили?

### Декларація про використання штучного інтелекту

Зазначте, чи користувались ШІ-помічником, у яких саме місцях і як перевіряли
отриманий код. Використання дозволене; **неперевірене** використання — ні.

In [ ]:
# ЧЕРНЕТКА ВІДПОВІДЕЙ. Числа підставляються зі змінних ноутбука, а текст перечитайте
# і перепишіть своїми словами: у звіті має бути ваша думка і ваші перевірки, а не чужі.
n_mask, n_drop = int(mask.sum()), int((mask & ~has_name).sum())
drop_types = ', '.join('%s (%d)' % (k, v) for k, v in
                       heritage[mask & ~has_name]['historic'].value_counts().head(3).items())
a30 = L_nn30 / L_op30 - 1
common = sorted(set(top10['name']) & set(top_naive['name']))
pct_nn = 100 * (tour_length(D10, nn10) / L_opt - 1)
pct_2o = 100 * (tour_length(D10, op10) / L_opt - 1)
cells31 = 2 ** 30 * 30
i_far = int(np.argmax(dist30)); far_name, far_d = pts30['name'].iloc[i_far], float(np.max(dist30))
n_same = len(set(dropped_same))
eq_text = ('десятка не змінилась: у цих даних результат стійкий до вибору ваг'
           if overlap_eq == N_SMALL else
           'десятка змінилась: ваги помітно впливають на результат')

REPORT = f"""
1. З {n_mask} об'єктів шару heritage відкинуто {n_drop} ({100*n_drop/n_mask:.1f}%) за правилом «немає name»;
   лишилось {len(cand)}. Серед відкинутих найчастіше: {drop_types}. Замовнику без пояснень цей фільтр
   віддавати не можна: безіменний об'єкт може бути значущим, а правило механічне. Лінійних об'єктів
   у кандидатах {len(lines)}, тож правило про їх виключення тут нічого не прибрало.

2. Сигнали й ваги: wikidata/wikipedia — {W['wiki']}; кількість мов у name:* (rank pct, без рідної name:uk) —
   {W['n_lang']}; tourism=museum/gallery/attraction — {W['visit']}; заповнений historic — {W['hist']}.
   Wiki найважча, бо це визнання поза OSM; мови — інтерес іноземців; «призначений для відвідування» слабший,
   бо так позначають і малозначущі заклади; historic найлегший, бо historic=memorial ставлять і на
   другорядні знаки. Площу свідомо не взято в рейтинг: вона є другим, наївним критерієм.
   При рівних вагах (по 0.25) збіг із основною десяткою {overlap_eq} з {N_SMALL}: {eq_text}.
   У Чернігові мало мовних тегів (лише name:pl, name:en, name:ru), тому сигнал мов слабший, ніж у більших містах.

3. Збіглось {overlap} з {N_SMALL} (до чищення дублікатів — {overlap_raw}); спільні: {', '.join(common)}.
   Обидва списки отримано з одних даних одним кодом без помилок, але вони майже різні: «найкращі» — це не
   властивість даних, а наслідок вибраного критерію. За площею у верхівці опинилися, зокрема,
   «Нудистський пляж» і «АТБ-Маркет» — обʼєкти, які як пам'ятку замовник навряд чи прийняв би.

4. Коефіцієнт обходу: {detour_factor(D10, E10):.2f} (мала задача), {detour_factor(D30, E30):.2f} (велика). Це на
   нижній межі типового для пішохідної мережі діапазону 1,2–1,6. Асиметрія D дорівнює
   {np.max(np.abs(D10 - D10.T)):.0f} м: пішохідна мережа неорієнтована, шлях туди й назад однаковий.
   Припущення, чому коефіцієнт малий (щільна мережа footway/path у центрі), я не перевіряв.

5. Найближчий сусід гірший за точний розв'язок на {pct_nn:.2f}% на 10 точках ({tour_length(D10, nn10):.0f} м
   проти {L_opt:.0f} м); 2-opt дав {tour_length(D10, op10):.0f} м ({pct_2o:.2f}% від оптимуму).
   На 30 точках найближчий сусід довший за 2-opt на {100*a30:.1f}% ({L_nn30:.0f} м проти {L_op30:.0f} м).

6. Для 31 точки таблиця Held–Karp має 2^30·30 = {cells31:.1e} клітин; при 8 байтах на клітину це
   {cells31*8/1e9:.0f} ГБ пам'яті й близько {2**30*30**2:.1e} операцій. Таблиця не вміщається в пам'ять
   звичайного комп'ютера: справа не в часі, а в обсязі пам'яті.

7. Що було не так з даними:
   (а) {n_drop} обʼєктів без name;
   (б) {len(dropped)} обʼєктів у топ-{N_LARGE} стояли ближче 60 м до вищого за рейтингом, і ще були однойменні
   копії: назва «Чорна могила» стоїть на кількох елементах OSM, а запис «Національний архітектурно-історичний
   заповідник «Чернігів стародавній»» (установа на площу всього старого міста) опинився за 23 м від
   Спасо-Преображенського собору. Я лишив вищий за рейтингом, нижчі прибрав і добрав наступних;
   (в) у «пам'ятки» потрапляє те, що пам'яткою для туриста не є: військова техніка (ЗіС-5, МіГ-21ПФ,
   Т-34-85, ІСУ-152), «Пневматичний тир», «Нудистський пляж», «АТБ-Маркет» з тегом historic=ruins.
   З 82 кандидатів у топ-30 неминуче входять слабкі об'єкти; я їх не виключав, але для замовника це треба
   вирішити окремо;
   (г) точка «{far_name}» прив'язалась до графа за {far_d:.0f} м: поруч немає пішохідної мережі, тож усі
   віддалі до неї занижені приблизно на цю величину;
   (д) частину пам'ятників названо у давальному відмінку («Івану Мазепі», «Богдану Хмельницькому») — це
   присвята, а не назва місця.
   [Перевірте на карті, чи справді об'єкти з (б) — одна пам'ятка, і допишіть свій висновок.]

Походження даних: знімок збудовано з локального extract OSM (chernihiv.osm.pbf, BBBike) скриптом
build_city_from_pbf.py, бо Overpass API був недоступний (тайм-аути, 502). Тому порівняння «живий запит проти
знімка» за вимірами цього ноутбука зробити неможливо: в COST усі шари мають спосіб «знімок». Межу міста
(83,8 км²) отримано з Nominatim скриптом get_city_boundary.py: перший запит через ox.geocode_to_gdf
повернув контур Чернігівської області (31 884 км²), що не обрізало дані по місту, тож його відкинуто.
Метри рахуються в EPSG:32636 (UTM 36N), бо Чернігів лежить на 31,3° сх.д.

Декларація про використання ШІ:
Я користувався Claude. З його допомогою написано: скрипти build_city_from_pbf.py і get_city_boundary.py.
Як перевірено: у shapely координати (lon, lat), у folium (lat, lon), у H3 (lat, lng);
використано features_from_polygon (osmnx 2.x); самоперевірки check_tour і check_snap проходять,
Held–Karp ≤ 2-opt ≤ найближчий сусід; різних вузлів стільки ж, скільки точок; маршрут по ребрах
обминає квартали й видимо відрізняється від прямих відрізків на карті.
Помилки помічника, які довелося виправляти: спершу він вибрав Львів замість мого варіанта (Чернігів, № 15)
і не спитав номер; а контур області замість міста помітили лише за площею 31 884 км².
"""
print(REPORT)



1. З 121 об'єктів шару heritage відкинуто 39 (32.2%) за правилом «немає name»;
   лишилось 82. Серед відкинутих найчастіше: cannon (12), memorial (11), ruins (5). Замовнику без пояснень цей фільтр
   віддавати не можна: безіменний об'єкт може бути значущим, а правило механічне. Лінійних об'єктів
   у кандидатах 0, тож правило про їх виключення тут нічого не прибрало.

2. Сигнали й ваги: wikidata/wikipedia — 0.4; кількість мов у name:* (rank pct, без рідної name:uk) —
   0.3; tourism=museum/gallery/attraction — 0.2; заповнений historic — 0.1.
   Wiki найважча, бо це визнання поза OSM; мови — інтерес іноземців; «призначений для відвідування» слабший,
   бо так позначають і малозначущі заклади; historic найлегший, бо historic=memorial ставлять і на
   другорядні знаки. Площу свідомо не взято в рейтинг: вона є другим, наївним критерієм.
   При рівних вагах (по 0.25) збіг із основною десяткою 10 з 10: десятка не змінилась: у цих даних результат стійкий до вибору ваг.
   У Чернігові мало мо